In [1]:
import math_toolkit
math_toolkit.activate()

# Playing curve plots as sound

`math_toolkit` can stream an ordinary scalar curve to the browser speaker. The same symbolic expression that draws the curve is sampled in small audio windows, so a plotted function can become a sound source without using Plotly trace data as audio.

This tutorial uses simple oscillator formulas. The horizontal variable `x` is interpreted as audio time in seconds, and the curve value is interpreted as the speaker amplitude.

## Start quietly

Audio examples should begin with a small gain. The formulas below use `gain=0.15`, which is enough to hear but leaves room for expressions whose values briefly exceed the usual `[-1, 1]` audio range.

Curve sound normalization is also enabled by default. It attenuates loud sampled chunks before manual `gain` is applied, but it does not make quiet curves louder. Use `curve.sound.normalization = False` or `curve.sound.play(normalization=False)` only when you deliberately want the raw sampled amplitude.

Browser audio may also require a user activation before playback starts. If a call to `sound.play()` does not produce sound, inspect `sound.state()`. A status of `needs_user_activation` means the browser blocked the speaker start.

## Normalization

`sound.normalization` is a boolean setting on each playable curve. The default `True` keeps unexpectedly large samples inside a conservative ceiling before your manual `gain` is multiplied in. Set it to `False` for raw mathematical output when you are already controlling amplitude yourself.

The next cells compare a raw sine at the default normalization ceiling with a four-times-louder formula normalized back to that same ceiling. They should sound the same when started from the legend buttons, because legend clicks use the default playback options.

In [2]:
normalization_fig = figure("playing: normalization", new=True)
normalization_fig.view.range = {"x": (0, 0.05)}
normalization_fig.show()

normalization_tone1 = normalization_fig.plot(
    0.95 * sin(2 * pi * 220 * x),
    x,
    name="raw_reference",
    label="raw 220 Hz reference at 0.95",
)
normalization_tone1.sound.normalization = False

normalization_tone2 = normalization_fig.plot(
    4 * sin(2 * pi * 220 * x),
    x,
    name="normalized",
    label="4x formula normalized",
)
normalization_tone2.sound.normalization = True


In [3]:
normalization_tone1.sound.play(normalization=False)
normalization_tone1.sound.state().normalization

False

In [4]:
normalization_tone2.sound.play(normalization=True)
normalization_tone2.sound.state().normalization

True

## A first sine tone

A pure sine wave at 220 Hz is the expression

$$\sin(2\pi \cdot 220 \cdot x).$$

The plot shows the curve over a short visual window. The sound stream samples the same mathematical source as a continuous audio signal.

Use an unbounded `x` domain for sounds you want to keep playing. A finite domain such as `(x, 0, 0.04)` is interpreted as a finite audio clip and stops at `x = 0.04`.

In [5]:
tone_fig = figure("playing: first tone")
tone_fig.show()
tone = tone_fig.plot(
    sin(2 * pi * 220 * x),
    x,
    name="tone",
    label="220 Hz sine",
    samples=1000,
)
tone_fig.view.range = {"x": (0, 0.04)}


In [6]:
tone.sound.play(gain=0.15)
tone.sound.state()

AudioPlaybackState(status='playing', figure_id=2, node_id=3, plot_name='tone', label='220 Hz sine', time=0.0, elapsed=0.0, sample_rate=48000, gain=0.15, normalization=True, needs_user_activation=False)

In [7]:
tone.sound.stop()
tone_fig.sound.state()

AudioPlaybackState(status='stopped', figure_id=2, node_id=3, plot_name='tone', label='220 Hz sine', time=0.0, elapsed=0.0, sample_rate=48000, gain=0.15, normalization=True, needs_user_activation=False)

## Controlling pitch with a parameter

Multiplying the frequency by `2**a` makes the slider parameter `a` control pitch in octaves:

$$\sin(2\pi \cdot 220 \cdot 2^a \cdot x).$$

Changing `a` by `1` doubles the frequency. Changing it by `1/12` moves by one equal-tempered semitone. During playback, slider changes affect subsequent audio chunks.

In [8]:
pitch_fig = figure("playing: pitch slider", new=True)
pitch_fig.show()
pitch_fig.parameters({a: {"value": 0, "min": -1, "max": 1, "step": 1 / 1200}})

pitch_tone = pitch_fig.plot(
    sin(2 * pi * 220 * (2**a) * x),
    x,
    name="pitch",
    label=r"$220\, 2^a \; \text{Hz}$",
    samples=1000,
)
pitch_fig.view.range = {"x": (0, 0.04)}


In [9]:
pitch_tone.sound.play(gain=0.15)
pitch_tone.sound.state()

AudioPlaybackState(status='playing', figure_id=3, node_id=4, plot_name='pitch', label='$220\\, 2^a \\; \\text{Hz}$', time=0.0, elapsed=0.0, sample_rate=48000, gain=0.15, normalization=True, needs_user_activation=False)

Move the `a` slider while the sound is playing. The pitch should change without restarting the plot. The audio node attempts a small phase shift when the sampled expression changes, which helps avoid hard clicks at chunk boundaries.

In [10]:
pitch_fig.sound.stop()
pitch_fig.sound.state()

AudioPlaybackState(status='stopped', figure_id=3, node_id=4, plot_name='pitch', label='$220\\, 2^a \\; \\text{Hz}$', time=0.0, elapsed=0.0, sample_rate=48000, gain=0.15, normalization=True, needs_user_activation=False)

## A moving pitch formula

Now put a sine wave inside the exponent. The parameter `a` sets a base octave shift, `b` controls the depth of the pitch movement, and `c` controls the rate of that movement:

$$\sin\left(2\pi \cdot 220 \cdot x \cdot 2^{a + b\sin(2\pi c x)}\right).$$

This is a compact way to make a visibly wiggling oscillator and an audibly moving pitch. It is not a physically exact phase integral, but it is a useful symbolic sound sketch.

In [11]:
vibrato_fig = figure("playing: moving pitch")
vibrato_fig.show()
vibrato_expr = sin(2 * pi * 220 * x * (2 ** (a + b * sin(2 * pi * c * x))))
vibrato_fig.parameters({
    a: {"value": 0, "min": -1, "max": 1, "step": 1 / 12, "label": "base"},
    b: {"value": 0.15, "min": 0, "max": 0.75, "step": 0.01, "label": "depth"},
    c: {"value": 5, "min": 0.25, "max": 12, "step": 0.25, "label": "rate"},
})
vibrato = vibrato_fig.plot(
    vibrato_expr,
    x,
    name="vibrato",
    label="moving pitch",
    samples=1600,
)
vibrato_fig.view.range = {"x":(0, 0.08)}


In [12]:
vibrato.sound.play(gain=0.15)
vibrato.sound.state()

AudioPlaybackState(status='playing', figure_id=4, node_id=5, plot_name='vibrato', label='moving pitch', time=0.0, elapsed=0.0, sample_rate=48000, gain=0.15, normalization=True, needs_user_activation=False)

In [13]:
vibrato.sound.stop()

## A folded pitch-shape sketch

The expression below is closer to the compact shape

`sin(220 * 2*pi * (2**(a + b*sin(2*pi*c*x))))`.

Here the carrier phase is not multiplied by `x`; the time dependence enters through the inner modulation. That makes it less like a standard oscillator and more like a folded waveshaping experiment.

In [14]:
folded_fig = figure("playing: folded shape")
folded_expr = sin(220 * 2 * pi * (2 ** (a + b * sin(2 * pi * c * x))))
folded_fig.parameters({
    a: {"value": 0, "min": -0.25, "max": 0.25, "step": 0.01, "label": "offset"},
    b: {"value": 0.1, "min": 0, "max": 0.5, "step": 0.01, "label": "depth"},
    c: {"value": 2, "min": 0.25, "max": 10, "step": 0.25, "label": "rate"},
})
folded = folded_fig.plot(
    folded_expr,
    x,
    name="folded",
    label="folded pitch shape",
    samples=2000,
)
folded_fig.view.x_range = (0, 1)
folded_fig.show()

In [15]:
folded.sound.play(gain=0.1)
folded.sound.state()

AudioPlaybackState(status='playing', figure_id=5, node_id=6, plot_name='folded', label='folded pitch shape', time=0.0, elapsed=0.0, sample_rate=48000, gain=0.1, normalization=True, needs_user_activation=False)

In [16]:
folded.sound.stop()

## One active sound per figure

A figure can contain several playable curves, but only one curve is the active sound source at a time. Starting a second curve stops the previous active sound in the same figure. Selecting `fig.sound.current` chooses a curve without starting playback.

In [17]:
choice_fig = figure("playing: choices")
low = choice_fig.plot(sin(2 * pi * 110 * x), x, name="low", label="110 Hz")
high = choice_fig.plot(sin(2 * pi * 440 * x), x, name="high", label="440 Hz")
choice_fig.view.x_range = (0, 0.05)
choice_fig.show()

choice_fig.sound.current = "low"
choice_fig.sound.resume()
choice_fig.sound.state()

AudioPlaybackState(status='playing', figure_id=6, node_id=7, plot_name='low', label='110 Hz', time=0.0, elapsed=0.0, sample_rate=48000, gain=1.0, normalization=True, needs_user_activation=False)

In [18]:
high.sound.play(gain=0.15)
choice_fig.sound.state()

AudioPlaybackState(status='playing', figure_id=6, node_id=8, plot_name='high', label='440 Hz', time=0.0, elapsed=0.0, sample_rate=48000, gain=0.15, normalization=True, needs_user_activation=False)

In [19]:
choice_fig.sound.stop()
choice_fig.sound.state()

AudioPlaybackState(status='stopped', figure_id=6, node_id=8, plot_name='high', label='440 Hz', time=0.0, elapsed=0.0, sample_rate=48000, gain=0.15, normalization=True, needs_user_activation=False)

## Debugging playback

`sound.state()` returns a frozen state record. The most useful fields while debugging are `status`, `plot_name`, `time`, `elapsed`, `sample_rate`, and `needs_user_activation`.

If the state says `playing` but no sound is audible, lower the formula complexity and try a pure sine first. If the state says `needs_user_activation`, the browser refused to start Web Audio from the current notebook action.

In [20]:
state = vibrato.sound.state()
state.status, state.plot_name, state.time, state.sample_rate, state.needs_user_activation

('stopped', 'vibrato', 0.0, 48000, False)

## Recap

A playable curve is still an ordinary `plot(...)` curve. The `sound` namespace starts, stops, pauses, resumes, seeks, and reports the audio state. Parameter sliders remain the source of mathematical variation, and the audio sampler follows those parameter values without reading Plotly traces.

Use low gain for experiments, start with pure sines, and inspect `sound.state()` whenever playback behavior is unclear.

## Related topics

- [Plotting symbolic mathematics](plotting.ipynb) for the figure and plot workflow.
- [plot](../library/plot.ipynb) for scalar curves and plot handles.
- [figure](../library/figure.ipynb) for figure creation, display, routing, views, and layout hooks.
- [Num](../library/Num.ipynb) for the symbolic-to-numeric boundary.